# 1. Analysis - Balance Sheet

## Notebook Summary

This notebook analyzes a single companyâ€™s balance sheet history from Financial Modeling Prep using the local `TICKER` parameter near the top of this notebook. It is organized into data loading and preparation, annual and quarterly summary snapshots, long-horizon balance sheet trend charts, and common-size, liquidity, and capital-structure diagnostics.

Run the notebook from top to bottom after updating the local `TICKER` parameter near the top of this notebook so the helper functions, balance sheet datasets, and all downstream tables and charts stay in sync.

## Data Loading and Preparation

These cells initialize the FMP helpers, normalize the ticker, load annual and quarterly balance sheet history, and calculate the derived liquidity, leverage, and common-size fields used throughout the notebook.

They also build the base annual and quarterly summary tables that the rest of the analysis depends on.

In [ ]:
# 2. Setup
from pathlib import Path
import sys

import pandas as pd

financial_statement_params = {
    "ticker_str": "SOXL",
    "annual_limit": 20,
    "quarterly_limit": 40,
}

TICKER = financial_statement_params["ticker_str"]
ANNUAL_LIMIT = financial_statement_params["annual_limit"]
QUARTERLY_LIMIT = financial_statement_params["quarterly_limit"]

# Seed the local package import when the notebook starts in a subfolder.
for _project_root_candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_project_root_candidate / "Quantapp" / "project.py").exists():
        if str(_project_root_candidate) not in sys.path:
            sys.path.insert(0, str(_project_root_candidate))
        break
else:
    raise RuntimeError("Could not locate the project root containing Quantapp.")

from Quantapp.project import ensure_project_root_on_path

PROJECT_ROOT = ensure_project_root_on_path()

from Quantapp.data import get_balance_sheet_history


TICKER

In [ ]:
# 3. Load balance sheet history through Quantapp.data
balance_sheet_history = get_balance_sheet_history(
    TICKER,
    annual_limit=ANNUAL_LIMIT,
    quarterly_limit=QUARTERLY_LIMIT,
)

SYMBOL = balance_sheet_history.symbol
company_name = balance_sheet_history.company_name
chart_label = balance_sheet_history.chart_label
annual_balance = balance_sheet_history.annual
quarterly_balance = balance_sheet_history.quarterly

annual_balance.tail()

In [ ]:
# 4. Review quarterly balance sheet history
quarterly_balance.tail()

In [ ]:
# 5. Build annual summary stats
from IPython.display import display

latest_annual = annual_balance.iloc[-1]
annual_summary = pd.DataFrame(
    [
        {"metric": "Latest annual total assets", "value": latest_annual["totalAssets"], "style": "currency", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual cash and short-term investments", "value": latest_annual["cashAndShortTermInvestments"], "style": "currency", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual total debt", "value": latest_annual["totalDebt"], "style": "currency", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual net debt", "value": latest_annual["netDebt"], "style": "currency", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual total equity", "value": latest_annual["totalEquity"], "style": "currency", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual current ratio", "value": latest_annual["currentRatio"], "style": "ratio", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual cash ratio", "value": latest_annual["cashRatio"], "style": "ratio", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual debt to equity", "value": latest_annual["debtToEquity"], "style": "ratio", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual equity ratio", "value": latest_annual["equityRatio"], "style": "percent", "asOf": latest_annual["date"].date()},
    ]
)
annual_summary_display = annual_summary.copy()
annual_summary_display["value"] = [format_metric_value(value, style) for value, style in zip(annual_summary_display["value"], annual_summary_display["style"])]
display(annual_summary_display.loc[:, ["metric", "value", "asOf"]])

In [ ]:
# 6. Build quarterly summary stats
from IPython.display import display

latest_quarter = quarterly_balance.iloc[-1]
quarterly_summary = pd.DataFrame(
    [
        {"metric": "Latest quarterly total assets", "value": latest_quarter["totalAssets"], "style": "currency", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest quarterly current assets", "value": latest_quarter["totalCurrentAssets"], "style": "currency", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest quarterly current liabilities", "value": latest_quarter["totalCurrentLiabilities"], "style": "currency", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest quarterly working capital", "value": latest_quarter["workingCapital"], "style": "currency", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest quarterly current ratio", "value": latest_quarter["currentRatio"], "style": "ratio", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest quarterly cash ratio", "value": latest_quarter["cashRatio"], "style": "ratio", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest quarterly total debt", "value": latest_quarter["totalDebt"], "style": "currency", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest quarterly net debt", "value": latest_quarter["netDebt"], "style": "currency", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest quarterly debt to equity", "value": latest_quarter["debtToEquity"], "style": "ratio", "asOf": latest_quarter["date"].date()},
    ]
)
quarterly_summary_display = quarterly_summary.copy()
quarterly_summary_display["value"] = [format_metric_value(value, style) for value, style in zip(quarterly_summary_display["value"], quarterly_summary_display["style"])]
display(quarterly_summary_display.loc[:, ["metric", "value", "asOf"]])

## Trend and Composition Charts

These cells move from summary tables into visualization. They cover annual and quarterly balance sheet scale, liquidity, composition, and growth so you can separate long-term capital structure drift from shorter-term shifts in working capital and leverage.

In [ ]:
# 7. Plot annual balance sheet trends
from Quantapp.visualization.views.single_asset_profile.valuation.fundamentals import plot_balance_sheet_trends

annual_balance_fig = plot_balance_sheet_trends(
    annual_balance,
    chart_label=chart_label,
    period_label="annual",
)
annual_balance_fig.show(config={"responsive": True, "displaylogo": False})

In [ ]:
# 8. Plot quarterly balance sheet trends
from Quantapp.visualization.views.single_asset_profile.valuation.fundamentals import plot_balance_sheet_trends

quarterly_balance_fig = plot_balance_sheet_trends(
    quarterly_balance,
    chart_label=chart_label,
    period_label="quarterly",
)
quarterly_balance_fig.show(config={"responsive": True, "displaylogo": False})

## Common-Size, Liquidity, and Capital Structure

These cells recast the balance sheet as a share of total assets, then separate liquidity from leverage so you can see whether changes are being driven by asset composition, working-capital pressure, or capital-structure decisions.

In [ ]:
# 9. Build common-size balance sheet views
from IPython.display import display

def build_common_size_balance_sheet(frame: pd.DataFrame, column_labels: pd.Series) -> pd.DataFrame:
    total_assets = frame["totalAssets"].replace(0, pd.NA)
    statement_rows = [
        ("Cash and short-term investments", frame["cashAndShortTermInvestments"].div(total_assets)),
        ("Receivables", frame["netReceivables"].div(total_assets)),
        ("Inventory", frame["inventory"].div(total_assets)),
        ("Total current assets", frame["currentAssetsPctAssets"]),
        ("Net PP&E", frame["propertyPlantEquipmentNet"].div(total_assets)),
        ("Long-term investments", frame["longTermInvestments"].div(total_assets)),
        ("Goodwill and intangibles", frame["goodwillAndIntangibleAssets"].div(total_assets)),
        ("Total non-current assets", frame["nonCurrentAssetsPctAssets"]),
        ("Total assets", pd.Series(1.0, index=frame.index)),
        ("Accounts payable", frame["accountPayables"].div(total_assets)),
        ("Short-term debt", frame["shortTermDebt"].div(total_assets)),
        ("Total current liabilities", frame["currentLiabilitiesPctAssets"]),
        ("Long-term debt", frame["longTermDebt"].div(total_assets)),
        ("Total debt", frame["totalDebtPctAssets"]),
        ("Total non-current liabilities", frame["nonCurrentLiabilitiesPctAssets"]),
        ("Total liabilities", frame["totalLiabilitiesPctAssets"]),
        ("Total equity", frame["totalEquityPctAssets"]),
    ]

    filtered_rows = []
    for label, values in statement_rows:
        if label in {"Total assets", "Total liabilities", "Total equity"} or values.fillna(0).abs().gt(1e-6).any():
            filtered_rows.append((label, values))

    statement = pd.DataFrame(
        [values.reset_index(drop=True).tolist() for _, values in filtered_rows],
        index=[label for label, _ in filtered_rows],
        columns=column_labels.reset_index(drop=True).tolist(),
    )
    return statement.dropna(how="all", axis=0)

annual_common_size_frame = annual_balance.tail(min(len(annual_balance), 6)).copy()
annual_common_size_labels = annual_common_size_frame["calendarYear"].astype("Int64").astype(str)
annual_common_size = build_common_size_balance_sheet(annual_common_size_frame, annual_common_size_labels)

quarterly_common_size_frame = quarterly_balance.tail(min(len(quarterly_balance), 8)).copy()
quarterly_common_size_labels = (
    quarterly_common_size_frame["date"].dt.year.astype(str)
    + " "
    + quarterly_common_size_frame["quarterLabel"].astype(str)
    + " ("
    + quarterly_common_size_frame["date"].dt.strftime("%b %Y")
    + ")"
)
quarterly_common_size = build_common_size_balance_sheet(quarterly_common_size_frame, quarterly_common_size_labels)

display(annual_common_size.style.format("{:.1%}").set_caption(f"{SYMBOL} annual common-size balance sheet (% of total assets)"))
display(quarterly_common_size.style.format("{:.1%}").set_caption(f"{SYMBOL} quarterly common-size balance sheet (% of total assets)"))

In [ ]:
# 10. Plot liquidity and capital structure
from Quantapp.visualization.views.single_asset_profile.valuation.fundamentals import plot_liquidity_capital_structure

balance_sheet_risk_fig = plot_liquidity_capital_structure(
    annual_balance,
    quarterly_balance,
    chart_label=chart_label,
)
balance_sheet_risk_fig.show(config={"responsive": True, "displaylogo": False})